# Dell Endorsement Event Study

This notebook takes the single-name idea out of the SPY policy-shock notebook and tests it separately. The question is not whether Dell dipped after Trump spoke. The question is whether `DELL` outperformed the broad market and the tech sector after Trump publicly praised the company.

That makes this an endorsement or attention-shock event study, not a pure hold-the-dip test.


## Imports


In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


## Event Calendar

These are the Dell-related Trump events I am testing. I am using reported public remarks, then anchoring each event to the trading date when the market could react.


In [ ]:
dell_events = pd.DataFrame([
    {
        "event_date": "2026-02-19",
        "market_date": "2026-02-19",
        "event_name": "first reported Trump Dell endorsement",
        "description": "Trump reportedly encouraged supporters to buy a Dell computer at a Georgia rally.",
        "source": "https://www.washingtonpost.com/politics/2026/05/28/dell-inks-97-billion-pentagon-contract-after-trump-acquires-stock-praises-company/",
    },
    {
        "event_date": "2026-05-08",
        "market_date": "2026-05-08",
        "event_name": "White House buy-a-Dell remark",
        "description": "Trump praised Dell at a White House event and told Americans to buy Dell computers.",
        "source": "https://www.investing.com/news/stock-market-news/dell-stock-rockets-as-trump-makes-bullish-comments-4673713",
    },
    {
        "event_date": "2026-07-06",
        "market_date": "2026-07-06",
        "event_name": "Trump Accounts Dell praise",
        "description": "Trump praised Michael and Susan Dell during a Trump Accounts ceremony.",
        "source": "https://www.investing.com/news/stock-market-news/why-is-dell-technologies-stock-rallying-today-93CH-4777206",
    },
])

dell_events["event_date"] = pd.to_datetime(dell_events["event_date"])
dell_events["market_date"] = pd.to_datetime(dell_events["market_date"])

dell_events


## Pull Dell and Benchmark Data

I compare Dell against `SPY` and `XLK`. `SPY` is the broad-market benchmark. `XLK` is a simple technology-sector benchmark. This helps separate a Dell-specific move from a normal market or tech rally.


In [ ]:
hold_periods = [5, 10, 20, 30, 60]

start_date = dell_events["market_date"].min() - pd.Timedelta(days=10)
end_date = dell_events["market_date"].max() + pd.Timedelta(days=max(hold_periods) * 2)

prices = yf.download(
    ["DELL", "SPY", "XLK"],
    start=start_date.strftime("%Y-%m-%d"),
    end=end_date.strftime("%Y-%m-%d"),
    auto_adjust=True,
    progress=False,
)["Close"]

prices.tail()


## Forward Returns

For each event, I buy on the first available trading date after the remark and calculate Dell's return after 5, 10, 20, 30, and 60 trading days. I also subtract the return of `SPY` and `XLK` over the same window.


In [ ]:
rows = []

for _, event in dell_events.iterrows():
    future_prices = prices.loc[event["market_date"]:]

    if future_prices.empty:
        continue

    buy_date = future_prices.index[0]
    buy_prices = future_prices.iloc[0]

    row = {
        "event_name": event["event_name"],
        "event_date": event["event_date"],
        "buy_date": buy_date,
        "dell_buy_price": buy_prices["DELL"],
    }

    for hold in hold_periods:
        if hold < len(future_prices):
            sell_prices = future_prices.iloc[hold]
            returns = sell_prices / buy_prices - 1

            row[f"{hold}d_dell_return"] = returns["DELL"]
            row[f"{hold}d_vs_spy"] = returns["DELL"] - returns["SPY"]
            row[f"{hold}d_vs_xlk"] = returns["DELL"] - returns["XLK"]
        else:
            row[f"{hold}d_dell_return"] = np.nan
            row[f"{hold}d_vs_spy"] = np.nan
            row[f"{hold}d_vs_xlk"] = np.nan

    rows.append(row)

dell_event_returns = pd.DataFrame(rows)

dell_event_returns


## Comparison Table

This table is the main output. The raw Dell return shows what happened if I bought Dell after the event. The benchmark columns show whether Dell beat `SPY` or `XLK` over the same holding period.


In [ ]:
comparison_cols = []

for hold in hold_periods:
    comparison_cols += [
        f"{hold}d_dell_return",
        f"{hold}d_vs_spy",
        f"{hold}d_vs_xlk",
    ]

dell_comparison = dell_event_returns.set_index("event_name")[comparison_cols]

dell_comparison.style.format("{:.2%}")


## Average Event Return Plot

This plot averages across the Dell events and compares Dell's raw forward return with its excess return over the broad market and tech sector.


In [ ]:
plot_data = pd.DataFrame({
    "DELL": [dell_event_returns[f"{hold}d_dell_return"].mean() for hold in hold_periods],
    "DELL minus SPY": [dell_event_returns[f"{hold}d_vs_spy"].mean() for hold in hold_periods],
    "DELL minus XLK": [dell_event_returns[f"{hold}d_vs_xlk"].mean() for hold in hold_periods],
}, index=hold_periods)

plot_data.plot(figsize=(10, 6), marker="o")
plt.axhline(0, color="black", linewidth=1)
plt.title("Average Dell Return After Trump Dell Remarks")
plt.xlabel("Holding period in trading days")
plt.ylabel("Average return")
plt.tight_layout()
plt.show()


## How I Would Interpret This

The chart shows average forward returns after the selected Trump-Dell remarks. The y-axis is in decimal return units, so `1.0` means a `100%` return, not `1%`. That means Dell went up a lot in this small event sample.

The raw `DELL` line shows what happened if I bought Dell after each event and held for 5, 10, 20, 30, or 60 trading days. The `DELL minus SPY` line subtracts the broad market return over the same window. The `DELL minus XLK` line subtracts the technology-sector return over the same window.

The important result is that Dell's benchmark-adjusted returns are also strongly positive. That means Dell did not just rise because the whole market rose, and it did not just rise because large-cap tech rose. In this sample, Dell outperformed both `SPY` and `XLK` after the selected Trump-Dell remarks.

I still would not claim that Trump's comments caused Dell to rise. There are only three events, and Dell had major company-specific catalysts at the same time, including AI server demand, earnings momentum, analyst upgrades, and government-contract news. The later events also happen during the same strong Dell trend, so the average can be heavily influenced by one broad momentum period.

So my conclusion is: the Dell extension shows unusually strong post-event Dell returns, even after adjusting for the market and tech sector. But this is best interpreted as a single-name political endorsement or attention-shock event study, not proof of a standalone trading strategy.